### Test Envirnment 

- Step 1 → API connection
- Step 2 → PDF text extraction
- Step 3 → Text chunking
- Step 4 → Embeddings API
- Step 5 → FAISS vector index
- Step 6 → Retrieval
- Step 7 → RAG prompt
- Step 8 → LLM answer

#### Step 1 → API connection

In [1]:
from dotenv import load_dotenv
import os
from huggingface_hub import InferenceClient

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN not found in .env")


client = InferenceClient(api_key=HF_TOKEN)


In [2]:
# messages = [
#     {   "role" : "user",
#         "content" : "explain what is RAG in one line ?"
#     }
# ]

# response = client.chat_completion(
#     model="MiniMaxAI/MiniMax-M2.5",
#     messages=messages,
#     max_tokens=50
# )

# print(response)

In [3]:
# import json 
# print(json.dumps(response, indent=4))


In [4]:
# print(response.choices[0].message.content)

#### Step 2 → PDF text extraction

In [5]:
import fitz #PyMuPdf
import os 

pdf_path = "D:/Code/Python/Intership_Learning/AI_Ml_Projects/RAG_Q&A_Bot/PDFs/Introduction to Outlier.docx.pdf"

doc = fitz.open(pdf_path) #open in PyMuPdf

pdf_text = ""

for page in doc:
    pdf_text+= page.get_text() + "\n"

page_no = doc.page_count
doc.close()

print(f"PDF Loaded Successfully. \nDocument Total Pages : {page_no}\nNumber of character extracted : {len(pdf_text)}\n")


PDF Loaded Successfully. 
Document Total Pages : 22
Number of character extracted : 18680



#### Note :
- Sometimes PyMuPDF DLL causes import errors on Windows 11. 

- To fix: go to Defender → Virus & Threat Protection → Ransomware Protection → Manage Ransomware Protection → Allow an app through Controlled Folder Access → add your python.exe file.

- Restart VS code editor or Kernal

#### Step 3 → Text chunking

In [8]:
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

In [9]:
from nltk.tokenize import sent_tokenize

def chunk_text(text, chunk_size=5, overlap=1):
    """
    Splits text into overlapping sentence chunks.

    text: full document string
    chunk_size: number of sentences per chunk
    overlap: number of sentences overlapping between chunks
    """

    sentences = sent_tokenize(text)

    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(sentences), step):
        chunk = " ".join(sentences[i:i + chunk_size])
        chunks.append(chunk)

    return chunks


# Create chunks
chunks = chunk_text(pdf_text, chunk_size=5, overlap=1)

print(f"PDF split into {len(chunks)} chunks.")
print(f"\nExample chunk:\n{chunks[0]}")

PDF split into 29 chunks.

Example chunk:
Introduction to Outlier 
An outlier is an observation that is unlike the other 
observations. It is rare, or distinct, or does not fit in some way. It is also called anomalies. Outliers are extreme values or unusual data points in a dataset 
that differ significantly from other observations. They are 
crucial to understand because they can affect model accuracy 
and lead to misleading insights if not properly addressed.


#### Step 4 → Embeddings API

In [10]:
# Check tourch usage GPU name 
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 Laptop GPU


#### Main GPU usage → my case NVDIA GPU

In [11]:
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

# Load model on GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

def get_embedding(text):
    # Returns embedding on CPU as numpy array
    embedding = embedding_model.encode(text, convert_to_numpy=True)
    return embedding

embeddings = [get_embedding(chunk) for chunk in chunks]

embeddings = np.vstack(embeddings).astype("float32")
print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: (29, 384)


#### Step 5 → FAISS vector index

In [12]:
import faiss
import numpy as np


faiss.normalize_L2(embeddings) #Normaize embeddings

# Create faiss Index -> faster find similar chunks
embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)

# Add embeddings to index
index.add(embeddings)

print(f"FAISS index created with {index.ntotal} vectors.")

FAISS index created with 29 vectors.


In [13]:
# User Question Embeddings :
question = input("Enter Question Here: ....\n")
print(question)
question_embedd = get_embedding(question)

question_embedd = np.array([question_embedd]).astype(np.float32)

faiss.normalize_L2(question_embedd)

What example is given for multivariate outliers involving height and weight?


In [14]:
# Search FAISS cosign similarity:
top_k = min(5, len(chunks))

distances, indices = index.search(question_embedd, top_k)
print(f"Top Chunk indexes: {indices}")

Top Chunk indexes: [[1 2 0 4 3]]


#### Step 6 → Retrieval

In [15]:
# Retrave Chunk :
retrived_chunk = [chunks[i] for i in indices[0]]

for i, chunk in enumerate(retrived_chunk):
    print(f"\nChunk {i+1}")
    print(chunk)


Chunk 1
They are 
crucial to understand because they can affect model accuracy 
and lead to misleading insights if not properly addressed. Types of Outliers 
●​ Univariate Outliers: These are unusual values in a single 
feature. For example, in a dataset of people’s ages, an age 
of 120 would be an outlier if most values are between 0 
and 100. ●​ Multivariate Outliers: These are unusual combinations of 
values across multiple features. For example, in a dataset 
of students' heights and weights, a student who is 6 feet 

tall and weighs 30 kg might be considered an outlier 
based on the combined information.

Chunk 2
For example, in a dataset 
of students' heights and weights, a student who is 6 feet 

tall and weighs 30 kg might be considered an outlier 
based on the combined information. Why Outliers Matter 
Outliers are important because they can: 
●​ Bias Analysis and Model Training: Many machine learning 
algorithms, such as linear regression and k-means 
clustering, are sensiti

#### Step 7 → RAG prompt

In [16]:
context = "\n\n".join(retrived_chunk)

prompt = f"""
You are a strict document QA system.

Rules:
1. Answer ONLY using the provided context.
2. Do NOT use outside knowledge.
3. If the exact answer is not present, reply: "Not found in document".
4. Do not infer or guess.

Context:
{context}

Question:
{question}

Answer:
"""


#### Step 8 → LLM answer

In [17]:
# Send Prompt 

from huggingface_hub import InferenceClient

client = InferenceClient(api_key=HF_TOKEN)

response = client.chat_completion(
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    messages=[
        {"role": "user", "content": prompt}
    ],
    max_tokens=300
)

print(response)


ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content='A student who is 6 feet tall and weighs 30 kg.', reasoning=None, tool_call_id=None, tool_calls=None), logprobs=None, content_filter_results={'hate': {'filtered': False}, 'self_harm': {'filtered': False}, 'sexual': {'filtered': False}, 'violence': {'filtered': False}, 'jailbreak': {'filtered': False, 'detected': False}, 'profanity': {'filtered': False, 'detected': False}})], created=1773392982, id='34e648bba6d3485aa875b0ced429e5b8', model='meta-llama/llama-3-8b-instruct', system_fingerprint='', usage=ChatCompletionOutputUsage(completion_tokens=15, prompt_tokens=859, total_tokens=874, prompt_tokens_details=None, completion_tokens_details=None), object='chat.completion')


In [18]:

print(response.choices[0].message.content)

A student who is 6 feet tall and weighs 30 kg.
